# Make Custom Datasets From CSV Files.
MTBench provides many tools to make data loading easy and hopefully. In addition to its rich built-in datasets, MTbench also supports to use your own custom datasets. In this notebook, we will demonstrate how to create custom datasets (from csv files) for machine learning solutions with MTBench. We will use a [Kaggle dataset](https://www.kaggle.com/competitions/airbnb-recruiting-new-user-bookings) as an example to illustrate the process step by step.

## Step 1: Download the Kaggle Dataset.

After joining the competition, you can download this kaggle dataset using the following command.

In [1]:
!kaggle competitions download -c airbnb-recruiting-new-user-bookings -p ./

 96%|████████████████████████████████████▍ | 62.0M/64.7M [00:00<00:00, 85.5MB/s]
100%|██████████████████████████████████████| 64.7M/64.7M [00:00<00:00, 78.6MB/s]


Extract the dataset to the airbnb/data folder

In [10]:
%%bash
unzip airbnb-recruiting-new-user-bookings.zip -d airbnb
for file in airbnb/*.zip; do unzip -d airbnb/data "$file"; done

Archive:  airbnb-recruiting-new-user-bookings.zip
  inflating: airbnb/age_gender_bkts.csv.zip  
  inflating: airbnb/countries.csv.zip  
  inflating: airbnb/sample_submission_NDF.csv.zip  


  inflating: airbnb/sessions.csv.zip  
  inflating: airbnb/test_users.csv.zip  
  inflating: airbnb/train_users_2.csv.zip  
Archive:  airbnb/age_gender_bkts.csv.zip
  inflating: airbnb/data/age_gender_bkts.csv  
Archive:  airbnb/countries.csv.zip
  inflating: airbnb/data/countries.csv  
Archive:  airbnb/sample_submission_NDF.csv.zip
  inflating: airbnb/data/sample_submission_NDF.csv  
Archive:  airbnb/sessions.csv.zip
  inflating: airbnb/data/sessions.csv  
Archive:  airbnb/test_users.csv.zip
  inflating: airbnb/data/test_users.csv  
Archive:  airbnb/train_users_2.csv.zip
  inflating: airbnb/data/train_users_2.csv  


You get some csv files listed below as the raw dataset. we will convert these into an MTBench dataset in the following steps.

In [11]:
!ls airbnb/data

age_gender_bkts.csv  sample_submission_NDF.csv	test_users.csv
countries.csv	     sessions.csv		train_users_2.csv


## Step 2: Process the dataset.

In [2]:
from pathlib import Path
import pandas as pd

Since Airbnb did not provide a validation set, we randomly sampled 10%/10% of the training set as the validation/test set.

In [18]:
folder_path = Path('airbnb/data')
pqt_out_path = Path('datasets/airbnb/')
if not pqt_out_path.exists():
    pqt_out_path.mkdir(parents=True)

sessions = pd.read_csv(Path(folder_path, "sessions.csv"))
age_gender_bkts = pd.read_csv(Path(folder_path, "age_gender_bkts.csv"))
countries = pd.read_csv(Path(folder_path, "countries.csv"))
countries.rename(columns=lambda x: x.strip(), inplace=True)
train_users = pd.read_csv(Path(folder_path, "train_users_2.csv"))
train_users['date_first_booking'] = pd.to_datetime(train_users['date_first_booking'], errors='coerce')

users = train_users.drop(columns=['country_destination'])
users.to_parquet(Path(pqt_out_path, "users.pqt"))

sessions.to_parquet(Path(pqt_out_path, "sessions.pqt"))
age_gender_bkts.to_parquet(Path(pqt_out_path, "age_gender_bkts.pqt"))
countries.to_parquet(Path(pqt_out_path, "countries.pqt"))

total_samples = len(train_users)
train_samples = int(total_samples * 0.8)
validation_samples = int(total_samples * 0.1)

train_set = train_users.sample(n=train_samples)
df = train_users.drop(train_set.index)
validation_set = df.sample(n=validation_samples)
test_set = df.drop(validation_set.index)

task_out_path = Path(pqt_out_path, "destination")
if not task_out_path.exists():
    task_out_path.mkdir()

train_set.rename(columns={'country_destination': 'pred_destination'}, inplace=True)
validation_set.rename(columns={'country_destination': 'pred_destination'}, inplace=True)
test_set.rename(columns={'country_destination': 'pred_destination'}, inplace=True)
train_set.to_parquet(Path(pqt_out_path, "destination/dest_train.pqt"))
validation_set.to_parquet(Path(pqt_out_path, "destination/dest_validation.pqt"))
test_set.to_parquet(Path(pqt_out_path, "destination/dest_test.pqt"))

## Step 3: Write metadata.yaml.

The metadata primarily serves two functions: 1. Describing the schema of the database. 2. Defining the target task.
In the metadata.yaml file, you need to specify the name, path, file format, and included columns for each table. For each column, you need to specify its type, such as primary key, foreign key (and the table it points to), or features of type float/text/category/datetime.

Below is an example of metadata.yaml:

In [ ]:
dataset_name: airbnb
tables:
  - name: Age_gender_bkts
    source: age_gender_bkts.pqt
    format: parquet
    columns:
      - name: age_bucket
        dtype: category
      - name: country_destination
        dtype: category
      - name: gender
        dtype: category
      - name: population_in_thousands
        dtype: float
  - name: User_id
    source: user_id.pqt
    format: parquet
    columns:
      - name: user_id
        dtype: primary_key
      - name: action
        dtype: category
      - name: action_type
        dtype: category
      - name: device_type
        dtype: category
      - name: secs_elapsed
        dtype: float
  - name: Countries
    source: countries.pqt
    format: parquet
    columns:
      - name: country_destination
        dtype: category
      - name: destination_language
        dtype: category
      - name: lat_destination
        dtype: float
      - name: lng_destination
        dtype: float
      - name: distance_km
        dtype: float
tasks:
  - name: destination
    source: destination/dest_{split}.pqt
    format: parquet
    columns:
      - name: id
        dtype: foreign_key
        link_to: User_id.user_id
      - name: date_first_booking
        dtype: datetime
      - name: gender
        dtype: category
      - name: age
        dtype: float
      - name: signup_method
        dtype: category
      - name: language
        dtype: category
      - name: first_device_type
        dtype: category
      - name: country_destination
        dtype: category
    time_column: date_account_created
    evaluation_metric: auroc
    target_column: country_destination
    target_table: User_id
    task_type: classification

## Step 4: Preprocessing.

In [19]:
!python -m tab2graph.main preprocess datasets/airbnb transform datasets/airbnb-single

[I][2024-04-11 10:16:58,preprocess.py:46] No solution configuration file provided. Use default configuration.
[D][2024-04-11 10:16:58,preprocess.py:52] Config:
{"transforms": [{"name": "handle_dummy_table", "config": {}}, {"name": "key_mapping", "config": {}}, {"name": "column_transform_chain", "config": {"transforms": [{"name": "canonicalize_numeric"}, {"name": "canonicalize_datetime"}, {"name": "featurize_datetime", "config": {"methods": ["YEAR", "MONTH", "DAY", "DAYOFWEEK", "TIMESTAMP"]}}, {"name": "norm_numeric"}, {"name": "remap_category"}, {"name": "glove_text_embedding"}]}}, {"name": "fill_timestamp", "config": {}}]}
[I][2024-04-11 10:16:58,preprocess.py:54] Loading data ...
[I][2024-04-11 10:17:00,preprocess.py:57] Creating preprocess ...
[D][2024-04-11 10:17:00,composite.py:38] [CanonicalizeNumericConfig(), CanonicalizeDatetimeConfig(), FeaturizeDatetimeTransformConfig(methods=[<DatetimeFeaturizeMethod.YEAR: 'YEAR'>, <DatetimeFeaturizeMethod.MONTH: 'MONTH'>, <DatetimeFeaturize

## Step 5: Train and evaluate single-table models.

In [ ]:
!mkdir workspace

You can use either a single-table solution or a multi-table solution (Graph Neural Network). Here, I'll illustrate a single-table solution. You can control the model hyperparameters by modifying the config file.

In [ ]:
!WANDB_DISABLED=True python -m tab2graph.main fit-tab datasets/airbnb-single destination tabnn -p workspace -c single-mlp.yaml